In [1]:
import argparse

import cv2
import mmcv
from mmcv.transforms import Compose
from mmengine.utils import track_iter_progress

from mmdet.apis import inference_detector, init_detector
from mmdet.registry import VISUALIZERS

from mmengine.visualization import Visualizer
import numpy as np
from torchvision.transforms import Compose, Normalize, ToTensor


/home/ren2/anaconda3/envs/mmdetection_glottis/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocess_image(img, mean, std):
    preprocessing = Compose([
        ToTensor(),
        Normalize(mean=mean, std=std)
    ])
    return preprocessing(img.copy()).unsqueeze(0)

In [3]:
def parse_args():
    parser = argparse.ArgumentParser(description='MMDetection video demo')
    # parser.add_argument('--img_path', default='/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/data/coco_ins/val2017/0.png', help='Image Path')
    # parser.add_argument('--config', default='/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/configs/rtmdet/zy_rtmdet-ins_tiny_8xb32-300e_coco.py', help='Config file')
    # parser.add_argument('--checkpoint', default='/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/work_dirs_ins/zy_rtmdet-ins_tiny_detcov0_MyBlock1_OnlyNeck_k=5_FLassign133Topk3/best_coco/bbox_mAP_epoch_27.pth', help='Checkpoint file')
    parser.add_argument(
        '--device', default='cuda:0', help='Device used for inference')
    parser.add_argument(
        '--score-thr', type=float, default=0.3, help='Bbox score threshold')
    parser.add_argument('--out', type=str, help='Output video file')
    parser.add_argument('--show', action='store_true', help='Show video')
    parser.add_argument(
        '--wait-time',
        type=float,
        default=1,
        help='The interval of show (s), 0 is block')
    # args = parser.parse_args()
    args = parser.parse_args(args=[])
    return args

In [ ]:
args = parse_args()
config = '/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/configs/rtmdet/zy_rtmdet-ins_tiny_8xb32-300e_coco.py'
checkpoint = '/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/work_dirs_ins/zy_rtmdet-ins_tiny_detcov0_MyBlock1_OnlyNeck_k=5_FLassign133Topk3/best_coco/bbox_mAP_epoch_27.pth'
img_path = '/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/data/coco_ins/val2017/0.png'
# build the model from a config file and a checkpoint file
# model = init_detector(args.config, args.checkpoint, device=args.device)
model = init_detector(config, checkpoint, args.device)
# build test pipeline
model.cfg.test_dataloader.dataset.pipeline[0].type = 'LoadImageFromNDArray'
test_pipeline = Compose(model.cfg.test_dataloader.dataset.pipeline)

# Read image and preprocess image
image = mmcv.imread(img_path, channel_order='rgb')
# Build Visualizer
# visualizer = Visualizer(image=image)
# init visualizer
visualizer = VISUALIZERS.build(model.cfg.visualizer)
image_norm = np.float32(image) / 255
input_tensor = preprocess_image(image_norm,
                            mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225])
# feat = model(input_tensor)[0]
feat = model.test_step(image)[0]

drawn_img = visualizer.draw_featmap(feat, image, channel_reduction='select_max')
visualizer.show(drawn_img)

In [12]:
from mmdet.apis import inference_detector, init_detector

config = '/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/configs/rtmdet/zy_rtmdet-ins_tiny_8xb32-300e_coco.py'
checkpoint = '/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/work_dirs_ins/zy_rtmdet-ins_tiny_detcov0_MyBlock1_OnlyNeck_k=5_FLassign133Topk3/best_coco/bbox_mAP_epoch_27.pth'
device='cuda:3'
model = init_detector(config, checkpoint, device=device)
print(model)


Loads checkpoint by local backend from path: /home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/work_dirs_ins/zy_rtmdet-ins_tiny_detcov0_MyBlock1_OnlyNeck_k=5_FLassign133Topk3/best_coco/bbox_mAP_epoch_27.pth
The model and loaded state dict do not match exactly

size mismatch for bbox_head.rtm_cls.0.weight: copying a param with shape torch.Size([5, 96, 1, 1]) from checkpoint, the shape in current model is torch.Size([1, 96, 1, 1]).
size mismatch for bbox_head.rtm_cls.0.bias: copying a param with shape torch.Size([5]) from checkpoint, the shape in current model is torch.Size([1]).
size mismatch for bbox_head.rtm_cls.1.weight: copying a param with shape torch.Size([5, 96, 1, 1]) from checkpoint, the shape in current model is torch.Size([1, 96, 1, 1]).
size mismatch for bbox_head.rtm_cls.1.bias: copying a param with shape torch.Size([5]) from checkpoint, the shape in current model is torch.Size([1]).
size mismatch for bbox_head.rtm_cls.2.weight: copying a param with shape torch.Size([5, 96

In [2]:
from typing import Optional, Sequence, Union
from mmdet.utils import get_test_pipeline_cfg
from mmcv.ops import RoIPool
import torch
import torch.nn as nn

ImagesType = Union[str, np.ndarray, Sequence[str], Sequence[np.ndarray]]

def get_data(model: nn.Module, imgs: ImagesType, test_pipeline: Optional[Compose] = None):
    """Inference image(s) with the detector.

    Args:
        model (nn.Module): The loaded detector.
        imgs (str, ndarray, Sequence[str/ndarray]):
           Either image files or loaded images.
        test_pipeline (:obj:`Compose`): Test pipeline.

    Returns:
        :obj:`DetDataSample` or list[:obj:`DetDataSample`]:
        If imgs is a list or tuple, the same length list type results
        will be returned, otherwise return the detection results directly.
    """

    if isinstance(imgs, (list, tuple)):
        is_batch = True
    else:
        imgs = [imgs]
        is_batch = False

    cfg = model.cfg

    if test_pipeline is None:
        cfg = cfg.copy()
        test_pipeline = get_test_pipeline_cfg(cfg)
        if isinstance(imgs[0], np.ndarray):
            # Calling this method across libraries will result
            # in module unregistered error if not prefixed with mmdet.
            test_pipeline[0].type = 'mmdet.LoadImageFromNDArray'

        test_pipeline = Compose(test_pipeline)

    if model.data_preprocessor.device.type == 'cpu':
        for m in model.modules():
            assert not isinstance(
                m, RoIPool
            ), 'CPU inference with RoIPool is not supported currently.'

    result_list = []
    for img in imgs:
        # prepare data
        if isinstance(img, np.ndarray):
            # TODO: remove img_id.
            data_ = dict(img=img, img_id=0)
        else:
            # TODO: remove img_id.
            data_ = dict(img_path=img, img_id=0)
        # build the data pipeline
        data_ = test_pipeline(data_)

        data_['inputs'] = [data_['inputs']]
        data_['data_samples'] = [data_['data_samples']]

    return data_
    #     # forward the model
    #     with torch.no_grad():
    #         results = model.test_step(data_)[0]

    #     result_list.append(results)

    # if not is_batch:
    #     return result_list[0]
    # else:
    #     return result_list


In [3]:
import mmcv
from mmcv.transforms import Compose
from mmengine.utils import track_iter_progress

from mmdet.apis import inference_detector, init_detector
from mmdet.registry import VISUALIZERS

from mmengine.visualization import Visualizer
import numpy as np
from torchvision.transforms import Compose, Normalize, ToTensor

import argparse
import cv2
import numpy as np
import torch
from torchvision import models
from pytorch_grad_cam import GradCAM, \
    HiResCAM, \
    ScoreCAM, \
    GradCAMPlusPlus, \
    AblationCAM, \
    XGradCAM, \
    EigenCAM, \
    EigenGradCAM, \
    LayerCAM, \
    FullGrad, \
    GradCAMElementWise


from pytorch_grad_cam import GuidedBackpropReLUModel
from pytorch_grad_cam.utils.image import show_cam_on_image, \
    deprocess_image, \
    preprocess_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from mmdet.apis import inference_detector, init_detector
from mmdet.utils import get_test_pipeline_cfg


def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--use-cuda', action='store_true', default=False,
                        help='Use NVIDIA GPU acceleration')
    parser.add_argument(
        '--image-path',
        type=str,
        default='/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/data/coco_ins/val2017/0.png',
        help='Input image path')
    parser.add_argument('--aug_smooth', action='store_true',
                        help='Apply test time augmentation to smooth the CAM')
    parser.add_argument(
        '--eigen_smooth',
        action='store_true',
        help='Reduce noise by taking the first principle componenet'
        'of cam_weights*activations')
    parser.add_argument('--method', type=str, default='gradcam',
                        choices=['gradcam', 'hirescam', 'gradcam++',
                                 'scorecam', 'xgradcam',
                                 'ablationcam', 'eigencam',
                                 'eigengradcam', 'layercam', 'fullgrad'],
                        help='Can be gradcam/gradcam++/scorecam/xgradcam'
                             '/ablationcam/eigencam/eigengradcam/layercam')

    args = parser.parse_args(args=[])
    args.use_cuda = args.use_cuda and torch.cuda.is_available()
    if args.use_cuda:
        print('Using GPU for acceleration')
    else:
        print('Using CPU for computation')

    return args


if __name__ == '__main__':
    """ python cam.py -image-path <path_to_image>
    Example usage of loading an image, and computing:
        1. CAM
        2. Guided Back Propagation
        3. Combining both
    """

    args = get_args()
    methods = \
        {"gradcam": GradCAM,
         "hirescam": HiResCAM,
         "scorecam": ScoreCAM,
         "gradcam++": GradCAMPlusPlus,
         "ablationcam": AblationCAM,
         "xgradcam": XGradCAM,
         "eigencam": EigenCAM,
         "eigengradcam": EigenGradCAM,
         "layercam": LayerCAM,
         "fullgrad": FullGrad,
         "gradcamelementwise": GradCAMElementWise}

    # model = models.resnet50(pretrained=True)
    config = '/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/configs/rtmdet/zy_rtmdet-ins_tiny_8xb32-300e_coco.py'
    checkpoint = '/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/work_dirs_ins/zy_rtmdet-ins_tiny_detcov0_MyBlock1_OnlyNeck_k=5_FLassign133Topk3/best_coco/bbox_mAP_epoch_27.pth'
    device='cuda:3'
    model = init_detector(config, checkpoint, device=device)
    model.cfg.test_dataloader.dataset.pipeline[0].type = 'LoadImageFromNDArray'
    test_pipeline = Compose(model.cfg.test_dataloader.dataset.pipeline)

    # Choose the target layer you want to compute the visualization for.
    # Usually this will be the last convolutional layer in the model.
    # Some common choices can be:
    # Resnet18 and 50: model.layer4
    # VGG, densenet161: model.features[-1]
    # mnasnet1_0: model.layers[-1]
    # You can print the model to help chose the layer
    # You can pass a list with several target layers,
    # in that case the CAMs will be computed per layer and then aggregated.
    # You can also try selecting all layers of a certain type, with e.g:
    # from pytorch_grad_cam.utils.find_layers import find_layer_types_recursive
    # find_layer_types_recursive(model, [torch.nn.ReLU])
    target_layers = [model.backbone.stage4]

    rgb_img = cv2.imread(args.image_path)
    # rgb_img = cv2.imread(args.image_path, 1)[:, :, ::-1]
    # rgb_img = np.float32(rgb_img) / 255
    # input_tensor = preprocess_image(rgb_img,
    #                                 mean=[0.485, 0.456, 0.406],
    #                                 std=[0.229, 0.224, 0.225])
    
    # data_ = dict(img=input_tensor, img_id=0)

    data_ = get_data(model, rgb_img, test_pipeline=test_pipeline)
    # We have to specify the target we want to generate
    # the Class Activation Maps for.
    # If targets is None, the highest scoring category (for every member in the batch) will be used.
    # You can target specific categories by
    # targets = [e.g ClassifierOutputTarget(281)]
    targets = None

    # Using the with statement ensures the context is freed, and you can
    # recreate different CAM objects in a loop.
    cam_algorithm = methods[args.method]
    with cam_algorithm(model=model,
                       target_layers=target_layers,
                       use_cuda=args.use_cuda) as cam:

        # AblationCAM and ScoreCAM have batched implementations.
        # You can override the internal batch size for faster computation.
        cam.batch_size = 32
        grayscale_cam = cam(input_tensor=data_,
                            targets=targets,
                            aug_smooth=args.aug_smooth,
                            eigen_smooth=args.eigen_smooth)

        # Here grayscale_cam has only one image in the batch
        grayscale_cam = grayscale_cam[0, :]

        cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

        # cam_image is RGB encoded whereas "cv2.imwrite" requires BGR encoding.
        cam_image = cv2.cvtColor(cam_image, cv2.COLOR_RGB2BGR)

    gb_model = GuidedBackpropReLUModel(model=model, use_cuda=args.use_cuda)
    gb = gb_model(input_tensor, target_category=None)

    cam_mask = cv2.merge([grayscale_cam, grayscale_cam, grayscale_cam])
    cam_gb = deprocess_image(cam_mask * gb)
    gb = deprocess_image(gb)

    cv2.imwrite(f'{args.method}_cam.jpg', cam_image)
    cv2.imwrite(f'{args.method}_gb.jpg', gb)
    cv2.imwrite(f'{args.method}_cam_gb.jpg', cam_gb)

Using CPU for computation
Loads checkpoint by local backend from path: /home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/work_dirs_ins/zy_rtmdet-ins_tiny_detcov0_MyBlock1_OnlyNeck_k=5_FLassign133Topk3/best_coco/bbox_mAP_epoch_27.pth
The model and loaded state dict do not match exactly

size mismatch for bbox_head.rtm_cls.0.weight: copying a param with shape torch.Size([5, 96, 1, 1]) from checkpoint, the shape in current model is torch.Size([1, 96, 1, 1]).
size mismatch for bbox_head.rtm_cls.0.bias: copying a param with shape torch.Size([5]) from checkpoint, the shape in current model is torch.Size([1]).
size mismatch for bbox_head.rtm_cls.1.weight: copying a param with shape torch.Size([5, 96, 1, 1]) from checkpoint, the shape in current model is torch.Size([1, 96, 1, 1]).
size mismatch for bbox_head.rtm_cls.1.bias: copying a param with shape torch.Size([5]) from checkpoint, the shape in current model is torch.Size([1]).
size mismatch for bbox_head.rtm_cls.2.weight: copying a param wi

TypeError: 'ConfigDict' object is not callable

In [2]:
import cv2

Img = cv2.imread('/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/data/coco_ins/train2017/5892.jpg',1)
print ('The number of lines of this image is:',Img.shape[0])
print ('The number of columns of this image is:',Img.shape[1])
print ('The number of channels of this image is:',Img.shape[2])
print ('The shape of  this image :',Img.shape)

[ WARN:0@52.066] global loadsave.cpp:244 findDecoder imread_('/home/ren2/data3/ZhangYang/mmdetection_v3x_glottis/data/coco_ins/train2017/5892.jpg'): can't open/read file: check file path/integrity


AttributeError: 'NoneType' object has no attribute 'shape'